# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2 Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided as a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access top-level metadata fields via the to_json() method
metadata = dataset.metadata.to_json()
print(f"{metadata['name']} (v{metadata['version']}): \n{metadata['description']}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List all record sets in the dataset metadata
record_sets = [rs['@id'] for rs in metadata['recordSet']]
print("Available Record Sets and their @id's:")
for idx, rs in enumerate(metadata['recordSet']):
    print(f"  {idx}. {rs['@id']} - name: {rs.get('name','(no name)')}")

# Optionally, show the fields/columns for each record set
for rs in metadata['recordSet']:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name','')}")
    print(f"  Description: {rs.get('description','')}")
    print("  Fields (by @id):")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            # If already dict (expanded)
            print(f"    - {field.get('@id')}: {field.get('name', '')}")
        else:
            print(f"    - {field}")

## 3. Data Extraction
Load data from selected record sets into DataFrames for analysis. All record sets, fields, and columns are referenced by their `@id` as per the Croissant model.

In [ ]:
# Select record sets by @id (example taking all listed in the metadata)
record_sets_ids = [rs['@id'] for rs in metadata['recordSet']]
dataframes = {}

for record_set_id in record_sets_ids:
    # Use the Croissant @id for extraction
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id}, shape: {df.shape}")
    except Exception as e:
        print(f"Could not load {record_set_id}: {e}")

# Show columns for the first record set (edit index if you want a different one)
main_recordset_id = record_sets_ids[0]
print(f"\nColumns for record set '@id': {main_recordset_id}")
print(dataframes[main_recordset_id].columns.tolist())
# Show sample of main table
dataframes[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering numeric field, normalization, grouping. All field references are by their Croissant `@id`. Adjust the selected fields as desired based on the overview above.

In [ ]:
# Identify a numeric field in this record set
main_df = dataframes[main_recordset_id]
# Choose a numeric field by @id; for illustration we'll pick a field containing 'coverage' or 'percent' in @id
import numpy as np
numeric_field_id = None
for c in main_df.columns:
    if any(x in c.lower() for x in ["coverage", "percent", "mw", "weight", "abundance", "count", "peptide", "score", "intensity"]):
        # Check if numeric field
        if np.issubdtype(main_df[c].dtype, np.number):
            numeric_field_id = c
            break
if numeric_field_id is None:
    # fallback: use first numeric field
    for c in main_df.columns:
        if np.issubdtype(main_df[c].dtype, np.number):
            numeric_field_id = c
            break
print(f"Using numeric field Croissant @id: {numeric_field_id}")

# Set threshold for filtering
threshold = main_df[numeric_field_id].mean() if numeric_field_id else 10
filtered_df = main_df[main_df[numeric_field_id] > threshold] if numeric_field_id else main_df.copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization
if numeric_field_id:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping: use first non-numeric field as group key
group_field = None
for c in main_df.columns:
    if not np.issubdtype(main_df[c].dtype, np.number):
        group_field = c
        break

if group_field and numeric_field_id:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize distribution of the selected numeric field and its relation to the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
if numeric_field_id and (numeric_field_id in main_df.columns):
    sns.histplot(main_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

if group_field and numeric_field_id:
    plt.figure(figsize=(14,4))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
This exploration notebook loaded and inspected the FAIR² Croissant dataset using only authoritative `@id` references for its record sets and fields. We reviewed structure, extracted records, filtered and normalized across a numeric column, and visualized distributions for initial analysis. You can extend this notebook with more advanced analyses as needed.